#Notebook 01: Single Cell Data Collection
Downloads GEO Series containing single-cell sequencing data for downstream analysis.

Dataset: GSE179640

Tan Y, Flynn WF, Sivajothi S, Luo D et al. Single-cell analysis of endometriosis reveals a coordinated transcriptional programme driving immunotolerance and angiogenesis across eutopic and ectopic tissues. Nat Cell Biol 2022 Aug;24(8):1306-1318. PMID: 35864314

In [31]:
# -- Installs
!pip -q install anndata scanpy session-info GEOparse pathlib

In [32]:
# -- Imports
import os

from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
import session_info
import GEOparse
import scanpy as sc


In [33]:
# -- Mount drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [34]:
# -- Paths
project_dir = Path(
    "/content/drive/MyDrive/endo-immune-atlas"
)

dataset = "GSE179640"

download_dir = Path("/content/geo_download")

raw_data_dir = (
    project_dir
    / "data"
    / "raw"
    / dataset
)

interim_data_dir = (
    project_dir
    / "data"
    / "interim"
    / dataset
)

output_file = (
    interim_data_dir
    / "raw.h5ad"
)

raw_data_dir.mkdir(
    parents=True,
    exist_ok=True,
)

interim_data_dir.mkdir(
    parents=True,
    exist_ok=True,
)

In [35]:
# -- Fetch GSE data from GEO
gse = GEOparse.get_GEO(
    geo="GSE179640",
    destdir=str(download_dir),
    include_data=True
)

gse.download_supplementary_files(directory=str(download_dir))

28-Jul-2026 02:49:22 DEBUG utils - Directory /content/geo_download already exists. Skipping.
DEBUG:GEOparse:Directory /content/geo_download already exists. Skipping.
28-Jul-2026 02:49:22 INFO GEOparse - File already exist: using local version.
INFO:GEOparse:File already exist: using local version.
28-Jul-2026 02:49:22 INFO GEOparse - Parsing /content/geo_download/GSE179640_family.soft.gz: 
INFO:GEOparse:Parsing /content/geo_download/GSE179640_family.soft.gz: 
28-Jul-2026 02:49:22 DEBUG GEOparse - DATABASE: GeoMiame
DEBUG:GEOparse:DATABASE: GeoMiame
28-Jul-2026 02:49:22 DEBUG GEOparse - SERIES: GSE179640
DEBUG:GEOparse:SERIES: GSE179640
28-Jul-2026 02:49:22 DEBUG GEOparse - PLATFORM: GPL24676
DEBUG:GEOparse:PLATFORM: GPL24676
28-Jul-2026 02:49:22 DEBUG GEOparse - SAMPLE: GSM6102532
DEBUG:GEOparse:SAMPLE: GSM6102532
28-Jul-2026 02:49:22 DEBUG GEOparse - SAMPLE: GSM6102533
DEBUG:GEOparse:SAMPLE: GSM6102533
28-Jul-2026 02:49:22 DEBUG GEOparse - SAMPLE: GSM6102534
DEBUG:GEOparse:SAMPLE: GSM

{'GSM6102532': {'ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6102nnn/GSM6102532/suppl/GSM6102532_C01_Ctrl_filtered_feature_bc_matrix.h5': '/content/geo_download/Supp_GSM6102532_Control_Patient_1_-_Control_Endometrium/GSM6102532_C01_Ctrl_filtered_feature_bc_matrix.h5'},
 'GSM6102533': {'ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6102nnn/GSM6102533/suppl/GSM6102533_C02_Ctrl_filtered_feature_bc_matrix.h5': '/content/geo_download/Supp_GSM6102533_Control_Patient_2_-_Control_Endometrium/GSM6102533_C02_Ctrl_filtered_feature_bc_matrix.h5'},
 'GSM6102534': {'ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6102nnn/GSM6102534/suppl/GSM6102534_C03_Ctrl_filtered_feature_bc_matrix.h5': '/content/geo_download/Supp_GSM6102534_Control_Patient_3_-_Control_Endometrium/GSM6102534_C03_Ctrl_filtered_feature_bc_matrix.h5'},
 'GSM6102537': {'ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6102nnn/GSM6102537/suppl/GSM6102537_E01_EuE_filtered_feature_bc_matrix.h5': '/content/geo_download/Supp_GSM6102537_Endometriosis_Patient_

In [36]:
# -- Check downloaded files
downloaded_files = sorted(
    path
    for path in download_dir.rglob("*")
    if path.is_file()
)

print(
    f"Downloaded files: {len(downloaded_files)}"
)

for file in downloaded_files:
    print(file)

Downloaded files: 64
/content/geo_download/GSE179640_family.soft.gz
/content/geo_download/Supp_GSM6102532_Control_Patient_1_-_Control_Endometrium/GSM6102532_C01_Ctrl_filtered_feature_bc_matrix.h5
/content/geo_download/Supp_GSM6102533_Control_Patient_2_-_Control_Endometrium/GSM6102533_C02_Ctrl_filtered_feature_bc_matrix.h5
/content/geo_download/Supp_GSM6102534_Control_Patient_3_-_Control_Endometrium/GSM6102534_C03_Ctrl_filtered_feature_bc_matrix.h5
/content/geo_download/Supp_GSM6102537_Endometriosis_Patient_1_-_Eutopic/GSM6102537_E01_EuE_filtered_feature_bc_matrix.h5
/content/geo_download/Supp_GSM6102540_Endometriosis_Patient_2_-_Eutopic/GSM6102540_E02_EuE_filtered_feature_bc_matrix.h5
/content/geo_download/Supp_GSM6102543_Endometriosis_Patient_3_-_Eutopic/GSM6102543_E03_EuE_filtered_feature_bc_matrix.h5
/content/geo_download/Supp_GSM6102546_Endometriosis_Patient_4_-_Eutopic/GSM6102546_E04_EuE_filtered_feature_bc_matrix.h5
/content/geo_download/Supp_GSM6102549_Endometriosis_Patient_5_-_

In [37]:
# -- Set up metadata

# samples to exclude
EXCLUDE = {"EOR", "EcPA"}

# metadata map keyed by sample_id extracted from filename
metadata = {
    # controls
    "GSM6102532_C01_Ctrl": {"patient_id": "C01", "tissue_type": "Ctrl",          "condition": "control",         "lesion_site": None},
    "GSM6102533_C02_Ctrl": {"patient_id": "C02", "tissue_type": "Ctrl",          "condition": "control",         "lesion_site": None},
    "GSM6102534_C03_Ctrl": {"patient_id": "C03", "tissue_type": "Ctrl",          "condition": "control",         "lesion_site": None},

    # eutopic
    "GSM6102537_E01_EuE":  {"patient_id": "E01", "tissue_type": "EuE",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102540_E02_EuE":  {"patient_id": "E02", "tissue_type": "EuE",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102543_E03_EuE":  {"patient_id": "E03", "tissue_type": "EuE",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102546_E04_EuE":  {"patient_id": "E04", "tissue_type": "EuE",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102549_E05_EuE":  {"patient_id": "E05", "tissue_type": "EuE",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102551_E06_EuE":  {"patient_id": "E06", "tissue_type": "EuE",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102554_E07_EuE":  {"patient_id": "E07", "tissue_type": "EuE",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102555_E08_EuE":  {"patient_id": "E08", "tissue_type": "EuE",          "condition": "endometriosis",   "lesion_site": None},
    "GSM6102560_E09_EuE":  {"patient_id": "E09", "tissue_type": "EuE",          "condition": "endometriosis",   "lesion_site": None},

    # ectopic peritoneal
    "GSM6595248_E01_EcP":  {"patient_id": "E01", "tissue_type": "EcP",          "condition": "endometriosis",   "lesion_site": "peritoneal"},
    "GSM6595250_E02_EcP":  {"patient_id": "E02", "tissue_type": "EcP",          "condition": "endometriosis",   "lesion_site": "peritoneal"},
    "GSM6595252_E03_EcP":  {"patient_id": "E03", "tissue_type": "EcP",          "condition": "endometriosis",   "lesion_site": "peritoneal"},
    "GSM6595254_E04_EcP":  {"patient_id": "E04", "tissue_type": "EcP",          "condition": "endometriosis",   "lesion_site": "peritoneal"},
    "GSM6595256_E05_EcP":  {"patient_id": "E05", "tissue_type": "EcP",          "condition": "endometriosis",   "lesion_site": "peritoneal"},
    "GSM6102550_E06_EcP":  {"patient_id": "E06", "tissue_type": "EcP",          "condition": "endometriosis",   "lesion_site": "peritoneal"},
    "GSM6102553_E07_EcP":  {"patient_id": "E07", "tissue_type": "EcP",          "condition": "endometriosis",   "lesion_site": "peritoneal"},
    "GSM6595258_E09_EcP":  {"patient_id": "E09", "tissue_type": "EcP",          "condition": "endometriosis",   "lesion_site": "peritoneal"},

    # ectopic ovarian
    "GSM6102552_E07_EcO":  {"patient_id": "E07", "tissue_type": "EcO",          "condition": "endometriosis",   "lesion_site": "ovarian"},
    "GSM6102556_E09_EcO":  {"patient_id": "E09", "tissue_type": "EcO",          "condition": "endometriosis",   "lesion_site": "ovarian"},
    "GSM6595261_E10_EcO":  {"patient_id": "E10", "tissue_type": "EcO",          "condition": "endometriosis",   "lesion_site": "ovarian"},
    "GSM6102562_E11_EcO":  {"patient_id": "E11", "tissue_type": "EcO",          "condition": "endometriosis",   "lesion_site": "ovarian"},
}

In [38]:
# -- Collect all wanted files
h5_files = sorted(
    download_dir.rglob(
        "*_filtered_feature_bc_matrix.h5"
    )
)

print(f"Found {len(h5_files)} H5 files.")

for file in h5_files:
    print(file.name)

Found 33 H5 files.
GSM6102532_C01_Ctrl_filtered_feature_bc_matrix.h5
GSM6102533_C02_Ctrl_filtered_feature_bc_matrix.h5
GSM6102534_C03_Ctrl_filtered_feature_bc_matrix.h5
GSM6102537_E01_EuE_filtered_feature_bc_matrix.h5
GSM6102540_E02_EuE_filtered_feature_bc_matrix.h5
GSM6102543_E03_EuE_filtered_feature_bc_matrix.h5
GSM6102546_E04_EuE_filtered_feature_bc_matrix.h5
GSM6102549_E05_EuE_filtered_feature_bc_matrix.h5
GSM6102550_E06_EcP_filtered_feature_bc_matrix.h5
GSM6102551_E06_EuE_filtered_feature_bc_matrix.h5
GSM6102552_E07_EcO_filtered_feature_bc_matrix.h5
GSM6102553_E07_EcP_filtered_feature_bc_matrix.h5
GSM6102554_E07_EuE_filtered_feature_bc_matrix.h5
GSM6102555_E08_EuE_filtered_feature_bc_matrix.h5
GSM6102556_E09_EcO_filtered_feature_bc_matrix.h5
GSM6102560_E09_EuE_filtered_feature_bc_matrix.h5
GSM6102562_E11_EcO_filtered_feature_bc_matrix.h5
GSM6102563_EOR01_filtered_feature_bc_matrix.h5
GSM6102565_EOR03_filtered_feature_bc_matrix.h5
GSM6595248_E01_EcP_filtered_feature_bc_matrix.h5
GS

In [41]:
# -- Set up objects
adatas = []

for file in h5_files:
    sample_id = file.name.replace("_filtered_feature_bc_matrix.h5", "")

    # -- Exclude organoids and unwanted tissue types (Adjacent peritoneum is excluded from v1)

    if any(excl in sample_id for excl in EXCLUDE):
      print(f"Skipping: {sample_id}")
      continue

    # -- Exclude samples without any metadata

    if sample_id not in metadata:
      print(
          f"Skipping {sample_id} - No metadata found."
          )
      continue

    # -- Load data
    adata = sc.read_10x_h5(file)
    adata.var_names_make_unique()

    # -- Add metadata to the object
    meta = metadata[sample_id]
    adata.obs["sample_id"] = sample_id
    adata.obs["patient_id"] = meta['patient_id']
    adata.obs["tissue_type"] = meta["tissue_type"]
    adata.obs["condition"] = meta["condition"]
    adata.obs["lesion_site"] = meta["lesion_site"]
    adata.obs["dataset"] = "GSE179640"

    # -- Append sample name to cell barcodes
    adata.obs_names = [
        f"{sample_id}_{bc}"
        for bc in adata.obs_names
    ]

    print(f"{sample_id} loaded - {adata.n_obs} total cells")
    adatas.append(adata)

/usr/local/lib/python3.12/dist-packages/scanpy/readwrite.py:248: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  adata = adata.copy()


GSM6102532_C01_Ctrl loaded - 4133 total cells


/usr/local/lib/python3.12/dist-packages/scanpy/readwrite.py:248: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  adata = adata.copy()


GSM6102533_C02_Ctrl loaded - 7801 total cells


/usr/local/lib/python3.12/dist-packages/scanpy/readwrite.py:248: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  adata = adata.copy()


...

/usr/local/lib/python3.12/dist-packages/scanpy/readwrite.py:248: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  adata = adata.copy()


GSM6595256_E05_EcP loaded - 4043 total cells
Skipping: GSM6595257_E05_EcPA


/usr/local/lib/python3.12/dist-packages/scanpy/readwrite.py:248: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  adata = adata.copy()


GSM6595258_E09_EcP loaded - 2518 total cells
Skipping: GSM6595259_E09_EcPA
Skipping: GSM6595260_E09_EcPA2
GSM6595261_E10_EcO loaded - 12788 total cells


/usr/local/lib/python3.12/dist-packages/scanpy/readwrite.py:248: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  adata = adata.copy()


In [42]:
# -- Check that samples were loaded
if not adatas:
  raise ValueError(
      "No samples were loaded. Check the GEO download directory and file names"
      )

print(f"\n Loaded {len(adatas)} samples")


 Loaded 24 samples


In [43]:
# -- Concat samples
combined = ad.concat(adatas, join = "outer")
combined.var_names_make_unique()

In [44]:
# -- Check concat was successful
print(f"Total cells: {combined.n_obs}")
print(f"Total genes: {combined.n_vars}")

print("\nCells by tissue type:")
print(combined.obs["tissue_type"].value_counts())

print("\nCells by lesion site:")
print(combined.obs["lesion_site"].value_counts(dropna=False))

Total cells: 119560
Total genes: 38224

Cells by tissue type:
tissue_type
EuE     41432
EcP     31664
EcO     28999
Ctrl    17465
Name: count, dtype: int64

Cells by lesion site:
lesion_site
None          58897
peritoneal    31664
ovarian       28999
Name: count, dtype: int64


In [45]:
# -- Check to make sure every donor is accounted for
donor_counts = (combined.obs
.groupby(
    ["tissue_type",
     "patient_id"], observed=True
    )
.size()
.unstack(fill_value=0)
)

donor_counts

patient_id,C01,C02,C03,E01,E02,E03,E04,E05,E06,E07,E08,E09,E10,E11
tissue_type,,,,,,,,,,,,,,
Ctrl,4133,7801,5531,0,0,0,0,0,0,0,0,0,0,0
EcO,0,0,0,0,0,0,0,0,0,2688,0,6152,12788,7371
EcP,0,0,0,2961,5049,6691,3536,4043,2494,4372,0,2518,0,0
EuE,0,0,0,7665,5277,5302,3819,3030,5226,3078,2780,5255,0,0


In [46]:
# -- Save to drive
output_file = (
    interim_data_dir
    / "raw.h5ad"
)

combined.write_h5ad(output_file)

In [47]:
## -- Run to see session info
session_info.show()

/usr/local/lib/python3.12/dist-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  mod_version = _find_version(mod.__version__)
/usr/local/lib/python3.12/dist-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  mod_version = _find_version(mod.__version__)
